In [1]:
import os
import json
import numpy as np
import pandas as pd

from flask import Flask, render_template, request
from werkzeug.utils import secure_filename

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

In [2]:
app = Flask(__name__)

In [3]:
# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------

PROJECT_PATH = r"C:\Users\Admin\Food and Calories estimation"

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "model",
    "food_model.keras"
)

CLASS_NAMES_PATH = os.path.join(
    PROJECT_PATH,
    "model",
    "class_names.json"
)

NUTRITION_PATH = r"C:\Users\Admin\Downloads\Indian Food Images\Indian_Food_Nutrition_51_Matching_Image_Classes.csv"

UPLOAD_FOLDER = os.path.join(
    PROJECT_PATH,
    "static",
    "uploads"
)

os.makedirs(UPLOAD_FOLDER, exist_ok=True)

app.config["UPLOAD_FOLDER"] = UPLOAD_FOLDER


In [4]:
# ------------------------------------------------------------
# 3. LOAD MODEL
# ------------------------------------------------------------

model = load_model(MODEL_PATH)

print("Model loaded successfully!")

Model loaded successfully!


In [5]:
# ------------------------------------------------------------
# 4. LOAD CLASS NAMES
# ------------------------------------------------------------

with open(CLASS_NAMES_PATH, "r") as f:
    class_names = json.load(f)

print("Class names loaded:", len(class_names))

Class names loaded: 51


In [6]:
# ------------------------------------------------------------
# 5. LOAD NUTRITION DATA
# ------------------------------------------------------------

nutrition = pd.read_csv(NUTRITION_PATH)

print("Nutrition dataset loaded!")
print("Nutrition records:", len(nutrition))

Nutrition dataset loaded!
Nutrition records: 51


In [7]:
# ------------------------------------------------------------
# 6. IMAGE SETTINGS
# ------------------------------------------------------------

IMAGE_SIZE = (224, 224)

In [8]:
# ------------------------------------------------------------
# 7. HOME PAGE
# ------------------------------------------------------------

@app.route("/")
def home():

    return render_template("index.html")

In [9]:
# ------------------------------------------------------------
# 8. PREDICTION
# ------------------------------------------------------------

@app.route("/predict", methods=["POST"])
def predict():

    # Check image
    if "image" not in request.files:
        return "No image uploaded."

    file = request.files["image"]

    if file.filename == "":
        return "No image selected."

    # Get serving size
    serving = float(request.form.get("serving", 100))

    # Secure filename
    filename = secure_filename(file.filename)

    # Save uploaded image
    image_path = os.path.join(
        app.config["UPLOAD_FOLDER"],
        filename
    )

    file.save(image_path)


    # --------------------------------------------------------
    # PREPROCESS IMAGE
    # --------------------------------------------------------

    img = image.load_img(
        image_path,
        target_size=IMAGE_SIZE
    )

    img_array = image.img_to_array(img)

    img_array = img_array / 255.0

    img_array = np.expand_dims(
        img_array,
        axis=0
    )


    # --------------------------------------------------------
    # PREDICT FOOD
    # --------------------------------------------------------

    prediction = model.predict(
        img_array,
        verbose=0
    )

    predicted_index = np.argmax(
        prediction[0]
    )

    food_name = class_names[predicted_index]

    confidence = (
        prediction[0][predicted_index] * 100
    )


    # --------------------------------------------------------
    # FIND NUTRITION INFORMATION
    # --------------------------------------------------------

    nutrition_row = nutrition[
        nutrition["food_name"] == food_name
    ]


    if nutrition_row.empty:

        return f"""
        <h2>Food recognized as: {food_name}</h2>
        <p>Nutrition information is not available.</p>
        <a href="/">Go Back</a>
        """


    nutrition_row = nutrition_row.iloc[0]


    # --------------------------------------------------------
    # CALCULATE NUTRITION FOR SERVING SIZE
    # --------------------------------------------------------

    multiplier = serving / 100.0


    calories = (
        nutrition_row["calories_per_100g"]
        * multiplier
    )

    protein = (
        nutrition_row["protein_g"]
        * multiplier
    )

    carbs = (
        nutrition_row["carbs_g"]
        * multiplier
    )

    fat = (
        nutrition_row["fat_g"]
        * multiplier
    )

    fiber = (
        nutrition_row["fiber_g"]
        * multiplier
    )


    # --------------------------------------------------------
    # DISPLAY RESULT
    # --------------------------------------------------------

    return render_template(
        "result.html",

        food_name=food_name,

        confidence=confidence,

        serving=serving,

        calories=calories,

        protein=protein,

        carbs=carbs,

        fat=fat,

        fiber=fiber,

        image_file=filename
    )



In [ ]:
# ------------------------------------------------------------
# 9. RUN APPLICATION
# ------------------------------------------------------------


if __name__ == "__main__":

    app.run(
        debug=True,
        use_reloader=False
    )

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [01/Oct/2026 09:05:48] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 09:05:48] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [01/Oct/2026 09:06:16] "POST /predict HTTP/1.1" 200 -
127.0.0.1 - - [01/Oct/2026 09:06:17] "GET /static/uploads/1ccf78d41c.jpg HTTP/1.1" 200 -
